# G6: repository activity profile (`06_repository_activity_profile`)

**Question.** For each repository and day: how much activity was there, and how is it split over the nine event types?

This notebook builds the Gold table `github_lakehouse.gold.repository_activity_profile`.

Contract: `docs/gold_contracts.md`, section 14. The way a Gold table is refreshed is in section 6.

| | |
|---|---|
| Reads | All nine Silver tables in `github_lakehouse.silver` |
| Writes | `gold.repository_activity_profile`, its staging copy `gold_staging.repository_activity_profile`, and one row in `audit.gold_pipeline_runs` |
| Grain | One row per repository per UTC day on which it had at least one event in the nine tables |
| Key | `repo_id`, `activity_date` |

The notebook has the same eleven steps as `04_repository_collaboration_metrics`. What is new: it reads **all nine** Silver tables, and it calculates nine shares that must add up to 100.

This notebook builds the **core columns** of the contract. The optional columns (active hours, push commit volume and the comment split) come in a second step.

## Nine kinds of events

| Kind | Silver table | Gold column |
|---|---|---|
| Push | `push_events` | `push_events` |
| Pull request | `pull_request_events` | `pull_request_events` |
| Issue | `issue_events` | `issue_events` |
| Issue comment | `issue_comment_events` | `issue_comment_events` |
| Review | `pull_request_review_events` | `review_events` |
| Inline review comment | `pull_request_review_comment_events` | `inline_review_comment_events` |
| Watch (a new star) | `watch_events` | `watch_events` |
| Fork | `fork_events` | `fork_events` |
| Release | `release_events` | `release_events` |

## The rules of this table

- **One event is one row and is counted once.** Each event belongs to exactly one kind, the kind of the Silver table it comes from. So the nine counts add up to the total.
- **A kind without events has the count 0 and the share 0.** That is a measured zero, not a missing value.
- **A share is 100 × the count of the kind ÷ all events of the row.** A row exists only when the repository has at least one event that day, so the total is never 0.
- **People are counted once across the nine tables.** Someone who pushes, comments and stars in the same repository on the same day is one actor.
- **An event without an actor** is an event, but not a person. The coverage column says which share of the events has a known actor.
- **Broader groups are not stored.** A report can add them from the nine counts, for example reviews plus inline review comments.

## The steps

| # | Step | What it does |
|---|---|---|
| 1 | Parameters | Reads the widgets and chooses the tables |
| 2 | Create the table | `CREATE TABLE IF NOT EXISTS` with the columns of the contract |
| 3 | Affected days | Decides which days this run rebuilds |
| 4 | Read Silver | Places the nine tables under each other, only the affected days |
| 5 | Build | The counts per repository and day, the name label, the shares and the coverage |
| 6 | Stage | Writes the result to the staging copy |
| 7 | Quality gates | Checks the staging copy and the event IDs |
| 8 | Publish | Replaces the affected days in the Gold table, only if the gates passed |
| 9 | Quality numbers | Totals for the audit row |
| 10 | Audit | Writes one audit row. Stops with an error if the run failed |
| 11 | Results | Shows the audit row and the Gold table |

Before the first run, `08_gold_audit_table` must have run once.

## 1. Parameters

The notebook has the three widgets every Gold notebook has.

| Widget | Blank (default) | With a value |
|---|---|---|
| `source_date` | All days are rebuilt and the whole Gold table is replaced | `YYYY-MM-DD`: only the days touched by that source date are rebuilt and replaced |
| `test_schema` | Production tables | `test_<name>`: every table is read from and written to `github_lakehouse.test_<name>` |
| `run_id` | A generated `manual-...` value | The Job passes its run ID |

### 1.1 Imports and fixed values

In [0]:
import re
import uuid
from datetime import datetime, timezone

# When this run started. It goes into the audit row.
run_started_at = datetime.now(timezone.utc)

# Every time in this notebook is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_NAME = "gold_repository_activity_profile"

### 1.2 `source_date`

Blank means all dates. A value must be a real date in the form `2024-02-03`. Anything else stops the notebook here, before it reads or writes a table.

In [0]:
dbutils.widgets.text("source_date", "")
source_date = dbutils.widgets.get("source_date").strip()

if source_date == "":
    source_date_value = None
    processing_scope = "ALL_DATES"
else:
    # strptime fails for text that is not a real date, for example 2024-02-31.
    try:
        source_date_value = datetime.strptime(source_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use a real date such as 2024-02-03, or leave it blank."
        )

    # strptime also accepts 2024-2-3. The text must look exactly like 2024-02-03.
    if str(source_date_value) != source_date:
        raise ValueError(
            f"Invalid source_date '{source_date}'. "
            "Use the form YYYY-MM-DD, for example 2024-02-03."
        )

    processing_scope = "SINGLE_DATE"

print("source_date:", source_date_value)
print("processing_scope:", processing_scope)

### 1.3 `test_schema` and the table names

The table names are set **only** in this cell: the schema of the nine Silver tables, the Gold table, its staging copy and the audit table. Later cells never change them, otherwise a test run could write to production.

The nine Silver tables are always in one schema. So the schema is chosen first, and the nine names are built from it.

A test schema name must start with `test_` and may contain only lower-case letters, digits and underscores.

In [0]:
dbutils.widgets.text("test_schema", "")
test_schema = dbutils.widgets.get("test_schema").strip()

if test_schema == "":
    SILVER_SCHEMA = "github_lakehouse.silver"
    GOLD_TABLE = "github_lakehouse.gold.repository_activity_profile"
    STAGING_TABLE = "github_lakehouse.gold_staging.repository_activity_profile"
    AUDIT_TABLE = "github_lakehouse.audit.gold_pipeline_runs"
else:
    if re.fullmatch("test_[a-z0-9_]+", test_schema) is None:
        raise ValueError(
            f"Invalid test_schema '{test_schema}'. "
            "It must start with test_ and may contain only lower-case letters, digits and underscores."
        )
    SILVER_SCHEMA = f"github_lakehouse.{test_schema}"
    GOLD_TABLE = f"github_lakehouse.{test_schema}.repository_activity_profile"
    STAGING_TABLE = f"github_lakehouse.{test_schema}.repository_activity_profile_staging"
    AUDIT_TABLE = f"github_lakehouse.{test_schema}.gold_pipeline_runs"

# The nine Silver tables.
PUSH_TABLE = SILVER_SCHEMA + ".push_events"
PULL_REQUEST_TABLE = SILVER_SCHEMA + ".pull_request_events"
ISSUE_TABLE = SILVER_SCHEMA + ".issue_events"
ISSUE_COMMENT_TABLE = SILVER_SCHEMA + ".issue_comment_events"
REVIEW_TABLE = SILVER_SCHEMA + ".pull_request_review_events"
REVIEW_COMMENT_TABLE = SILVER_SCHEMA + ".pull_request_review_comment_events"
WATCH_TABLE = SILVER_SCHEMA + ".watch_events"
FORK_TABLE = SILVER_SCHEMA + ".fork_events"
RELEASE_TABLE = SILVER_SCHEMA + ".release_events"

print("Silver schema :", SILVER_SCHEMA)
print("Gold table    :", GOLD_TABLE)
print("Staging table :", STAGING_TABLE)
print("Audit table   :", AUDIT_TABLE)

### 1.4 `run_id`

The Job passes its run ID. A run by hand gets a generated ID. The ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`.

In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip()

if run_id == "":
    run_id = "manual-" + str(uuid.uuid4())

if re.fullmatch("[A-Za-z0-9_-]+", run_id) is None:
    raise ValueError(
        f"Invalid run_id '{run_id}'. "
        "It may contain only letters, digits, - and _."
    )

print("run_id:", run_id)

### 1.5 Is the setup done?

The audit table is created by `08_gold_audit_table`. The nine Silver tables are created by the Silver notebooks `01` to `09`. If one of the ten tables does not exist, the notebook stops here with a clear message.

In [0]:
if not spark.catalog.tableExists(AUDIT_TABLE):
    raise RuntimeError(
        f"{AUDIT_TABLE} does not exist. Run 08_gold_audit_table first."
    )

silver_tables = [
    PUSH_TABLE,
    PULL_REQUEST_TABLE,
    ISSUE_TABLE,
    ISSUE_COMMENT_TABLE,
    REVIEW_TABLE,
    REVIEW_COMMENT_TABLE,
    WATCH_TABLE,
    FORK_TABLE,
    RELEASE_TABLE,
]

for silver_table in silver_tables:
    if not spark.catalog.tableExists(silver_table):
        raise RuntimeError(
            f"{silver_table} does not exist. Run the Silver notebook of this table first."
        )

print("Setup found  :", AUDIT_TABLE)
print("Sources found:")
for silver_table in silver_tables:
    print("  ", silver_table)

## 2. Create the Gold table

The columns, their types and their order are those of the contract. `CREATE TABLE IF NOT EXISTS` does nothing when the table already exists, so this step is safe to run again.

| Column | Meaning |
|---|---|
| `repo_id` | Key |
| `activity_date` | Key. The UTC date of the events |
| `repo_name` | Label: the latest name seen that day |
| `all_retained_event_count` | All events of the nine kinds that day |
| `push_events` to `release_events` | Nine columns, one per kind: the events of that kind. They add up to the total |
| `push_share_pct` to `release_share_pct` | Nine columns, one per kind: 100 × the count of the kind ÷ the total |
| `actor_known_events` | Events with a known actor. The base of the two columns below |
| `unique_known_actors` | Different actors across the nine kinds of events |
| `actor_attribution_coverage_pct` | 100 × events with a known actor ÷ all events |
| `_gold_run_id` | The run that last wrote the row |
| `_gold_updated_at` | When that run wrote it |

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
        -- Key and label
        repo_id                           BIGINT,
        activity_date                     DATE,
        repo_name                         STRING,

        -- All events, and the events per kind
        all_retained_event_count          BIGINT,
        push_events                       BIGINT,
        pull_request_events               BIGINT,
        issue_events                      BIGINT,
        issue_comment_events              BIGINT,
        review_events                     BIGINT,
        inline_review_comment_events      BIGINT,
        watch_events                      BIGINT,
        fork_events                       BIGINT,
        release_events                    BIGINT,

        -- The share of each kind, from 0 to 100
        push_share_pct                    DOUBLE,
        pull_request_share_pct            DOUBLE,
        issue_share_pct                   DOUBLE,
        issue_comment_share_pct           DOUBLE,
        review_share_pct                  DOUBLE,
        inline_review_comment_share_pct   DOUBLE,
        watch_share_pct                   DOUBLE,
        fork_share_pct                    DOUBLE,
        release_share_pct                 DOUBLE,

        -- The people
        actor_known_events                BIGINT,
        unique_known_actors               BIGINT,
        actor_attribution_coverage_pct    DOUBLE,

        -- Technical columns
        _gold_run_id                      STRING,
        _gold_updated_at                  TIMESTAMP
    )
    USING DELTA
""")

print("Table ready:", GOLD_TABLE)

## 3. Affected days

A Gold row is a summary of a whole day. When one new event arrives for a day, the row of that day has to be calculated again from **all** events of that day. So the first question of every run is: which days does this run have to rebuild?

### 3.1 The Silver rows in the scope of this run

- With a `source_date`, the scope is the Silver rows that were loaded from that source date.
- With a blank date, the scope is all Silver rows.

This table reads nine Silver tables, so the scope is taken from all nine. `UNION ALL` places the results of the nine queries under each other. Only the event date is needed here.

A row without an event date cannot be placed on a day. It is left out and counted here, and the count goes into the audit row. The profiling run found 0 such rows.

In [0]:
if source_date_value is None:
    scope_filter = "true"
else:
    scope_filter = f"_source_date = DATE'{source_date}'"

scope_rows = spark.sql(f"""
    SELECT event_date FROM {PUSH_TABLE}           WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {PULL_REQUEST_TABLE}   WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {ISSUE_TABLE}          WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {ISSUE_COMMENT_TABLE}  WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {REVIEW_TABLE}         WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {REVIEW_COMMENT_TABLE} WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {WATCH_TABLE}          WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {FORK_TABLE}           WHERE {scope_filter}
    UNION ALL
    SELECT event_date FROM {RELEASE_TABLE}        WHERE {scope_filter}
""")
scope_rows.createOrReplaceTempView("scope_rows")

scope_row = spark.sql("""
    SELECT
        COUNT(*)                      AS rows_in_scope,
        count_if(event_date IS NULL)  AS rows_without_event_date
    FROM scope_rows
""").collect()[0]

rows_in_scope = scope_row["rows_in_scope"]
rows_without_event_date = scope_row["rows_without_event_date"]

print("Scope filter                :", scope_filter)
print("Silver rows in the scope    :", rows_in_scope)
print("Rows without an event date  :", rows_without_event_date)

### 3.2 The days these rows belong to

The affected days are the event dates of the rows in the scope, from any of the nine tables. A day is rebuilt also when only one of the nine tables has a new row for it.

In the week that is loaded, the event date and the source date of a row are always the same day (measured by the profiling run). So a run for one source date rebuilds exactly one day.

In [0]:
day_rows = spark.sql("""
    SELECT DISTINCT event_date
    FROM scope_rows
    WHERE event_date IS NOT NULL
    ORDER BY event_date
""").collect()

affected_dates = []
for day_row in day_rows:
    affected_dates.append(day_row["event_date"])

print("Affected days:", len(affected_dates))
for affected_date in affected_dates:
    print("  ", affected_date)

### 3.3 The day list as SQL text

The later steps need the affected days inside a SQL `IN (...)` list, for example `DATE'2024-02-01', DATE'2024-02-02'`. The same text is used to read Silver, to check the result and to publish, so all three work on exactly the same days.

When there is no affected day, the text is `NULL`. `IN (NULL)` is never true, so the later queries simply find no row.

In [0]:
if len(affected_dates) == 0:
    affected_days_sql = "NULL"
else:
    day_texts = []
    for affected_date in affected_dates:
        day_texts.append("DATE'" + str(affected_date) + "'")
    affected_days_sql = ", ".join(day_texts)

print("Day list:", affected_days_sql)

## 4. Read Silver

All Silver rows of the affected days are read, **not only the rows of this source date**. A row that arrived earlier for the same day must be counted again, otherwise the new Gold row would be too small.

The nine tables are placed under each other with `UNION ALL`. They are **not joined**: a join of raw events from two tables would multiply the rows.

`UNION ALL` matches the columns by their position, not by their name. So the nine queries select the same seven columns in the same order:

| Column | Where it comes from |
|---|---|
| `event_kind` | A fixed text that says which table the row comes from, for example `'push'` |
| `event_id` | Silver. Used to check that no event is counted twice |
| `event_timestamp` | Silver. Used to find the latest name of the day |
| `activity_date` | Silver `event_date`, with its Gold name |
| `repo_id` | Silver. The key together with the day |
| `repo_name` | Silver. The label |
| `actor_id` | Silver. Used to count the people |

Every Silver table has these columns, so no column has to be filled with NULL here. The columns that only one table has, such as the commit count of a push, are not needed for the core columns.

Spark does not read anything yet. A temporary view is only a name for a query. The work is done in step 6, when the result is written.

In [0]:
activity_rows = spark.sql(f"""
    SELECT 'push' AS event_kind, event_id, event_timestamp, event_date AS activity_date, repo_id, repo_name, actor_id
    FROM {PUSH_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT 'pull_request', event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {PULL_REQUEST_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT 'issue', event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {ISSUE_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT 'issue_comment', event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {ISSUE_COMMENT_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT 'review', event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {REVIEW_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT 'review_comment', event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {REVIEW_COMMENT_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT 'watch', event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {WATCH_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT 'fork', event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {FORK_TABLE}
    WHERE event_date IN ({affected_days_sql})

    UNION ALL

    SELECT 'release', event_id, event_timestamp, event_date, repo_id, repo_name, actor_id
    FROM {RELEASE_TABLE}
    WHERE event_date IN ({affected_days_sql})
""")
activity_rows.createOrReplaceTempView("activity_rows")

The first of the nine queries gives the columns their names. The other eight only have to select the same columns in the same order.

### 4.1 How many Silver rows these days have

The rows are counted directly on each Silver table, not on the view above. So these numbers do not depend on the steps that follow.

The query below has nine small queries in brackets. Each one counts the rows of one Silver table and gives one number. Together they make one row with nine columns.

The nine numbers go into the audit row, one number per table. The quality gates compare each of them with the sum of its Gold column: every Silver row of the affected days must be counted exactly once.

In [0]:
count_row = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {PUSH_TABLE}           WHERE event_date IN ({affected_days_sql})) AS push_rows,
        (SELECT COUNT(*) FROM {PULL_REQUEST_TABLE}   WHERE event_date IN ({affected_days_sql})) AS pull_request_rows,
        (SELECT COUNT(*) FROM {ISSUE_TABLE}          WHERE event_date IN ({affected_days_sql})) AS issue_rows,
        (SELECT COUNT(*) FROM {ISSUE_COMMENT_TABLE}  WHERE event_date IN ({affected_days_sql})) AS issue_comment_rows,
        (SELECT COUNT(*) FROM {REVIEW_TABLE}         WHERE event_date IN ({affected_days_sql})) AS review_rows,
        (SELECT COUNT(*) FROM {REVIEW_COMMENT_TABLE} WHERE event_date IN ({affected_days_sql})) AS review_comment_rows,
        (SELECT COUNT(*) FROM {WATCH_TABLE}          WHERE event_date IN ({affected_days_sql})) AS watch_rows,
        (SELECT COUNT(*) FROM {FORK_TABLE}           WHERE event_date IN ({affected_days_sql})) AS fork_rows,
        (SELECT COUNT(*) FROM {RELEASE_TABLE}        WHERE event_date IN ({affected_days_sql})) AS release_rows
""").collect()[0]

# One number per Silver table. The names are the names of the Silver tables.
silver_row_counts = {
    "push_events": count_row["push_rows"],
    "pull_request_events": count_row["pull_request_rows"],
    "issue_events": count_row["issue_rows"],
    "issue_comment_events": count_row["issue_comment_rows"],
    "pull_request_review_events": count_row["review_rows"],
    "pull_request_review_comment_events": count_row["review_comment_rows"],
    "watch_events": count_row["watch_rows"],
    "fork_events": count_row["fork_rows"],
    "release_events": count_row["release_rows"],
}

silver_row_count = sum(silver_row_counts.values())

print("Silver rows on the affected days, per table:")
for silver_table_name in silver_row_counts:
    print("  ", silver_table_name, ":", silver_row_counts[silver_table_name])
print("All nine together:", silver_row_count)

## 5. Build

The Gold rows are built in three small queries. Each one is saved as a temporary view, and a later one reads it.

| Query | View | One row per |
|---|---|---|
| 5.1 | `activity_counts` | Repository and day: all the counts |
| 5.2 | `repo_names` | Repository and day: the name label |
| 5.3 | `gold_rows` | Repository and day: everything together, with the shares and the coverage |

### 5.1 The counts per repository and day

One `GROUP BY` over the stacked events gives one row per repository and day.

- `COUNT(*)` counts every event of the repository on that day.
- `count_if(event_kind = 'push')` counts only the pushes. A kind without events gets 0, not NULL.
- `COUNT(actor_id)` counts only the events where the actor is known.
- `COUNT(DISTINCT actor_id)` counts the different actors and leaves NULL out. With no known actor it gives 0.

**People.** `COUNT(DISTINCT actor_id)` runs over all nine kinds of events at once. That is why the tables were stacked first: someone who pushes and comments on the same day is counted once.

In [0]:
activity_counts = spark.sql("""
    SELECT
        repo_id,
        activity_date,

        COUNT(*)                                  AS all_retained_event_count,

        count_if(event_kind = 'push')             AS push_events,
        count_if(event_kind = 'pull_request')     AS pull_request_events,
        count_if(event_kind = 'issue')            AS issue_events,
        count_if(event_kind = 'issue_comment')    AS issue_comment_events,
        count_if(event_kind = 'review')           AS review_events,
        count_if(event_kind = 'review_comment')   AS inline_review_comment_events,
        count_if(event_kind = 'watch')            AS watch_events,
        count_if(event_kind = 'fork')             AS fork_events,
        count_if(event_kind = 'release')          AS release_events,

        COUNT(actor_id)                           AS actor_known_events,
        COUNT(DISTINCT actor_id)                  AS unique_known_actors

    FROM activity_rows
    GROUP BY repo_id, activity_date
""")
activity_counts.createOrReplaceTempView("activity_counts")

### 5.2 The `repo_name` label

A repository can be renamed, so `repo_name` is only a label and never a key. The label of a day is the name on the **latest** event of that repository on that day, whichever of the nine tables it comes from.

`ROW_NUMBER` numbers the events of each repository and day, the latest first. Number 1 is the one we keep.

- Events without a name are left out first, so a missing name is skipped.
- If two events have the same time, the one with the larger `event_id` gets number 1. `event_id` is text and is compared as text. This makes the result the same on every run.

In [0]:
repo_names = spark.sql("""
    SELECT
        repo_id,
        activity_date,
        repo_name
    FROM (
        SELECT
            repo_id,
            activity_date,
            repo_name,
            ROW_NUMBER() OVER (
                PARTITION BY repo_id, activity_date
                ORDER BY event_timestamp DESC, event_id DESC
            ) AS name_rank
        FROM activity_rows
        WHERE repo_name IS NOT NULL
    ) ranked_names
    WHERE name_rank = 1
""")
repo_names.createOrReplaceTempView("repo_names")

### 5.3 The Gold rows

The counts and the label are put together. The columns are in the order of the Gold table.

- `LEFT JOIN` keeps a repository-day whose events all have no name. Its label is NULL.
- **Share:** 100 × the count of one kind ÷ all events, rounded to 2 decimals. In SQL, `/` gives a decimal number also when both sides are whole numbers.
- The `CASE` keeps the query from dividing by zero. A row always has at least one event, so a share is never NULL here. A kind without events has the share 0.
- Each share is rounded on its own. So the nine shares of a row add up to 100 only within rounding, for example to 99.99.
- **Coverage:** 100 × events with a known actor ÷ all events, rounded to 2 decimals. A coverage of 0 means: the day has events, but none of them has a known actor. Then the number of people is 0 too.
- The two technical columns say which run wrote the row and when.

In [0]:
gold_rows = spark.sql(f"""
    SELECT
        c.repo_id,
        c.activity_date,
        n.repo_name,

        c.all_retained_event_count,
        c.push_events,
        c.pull_request_events,
        c.issue_events,
        c.issue_comment_events,
        c.review_events,
        c.inline_review_comment_events,
        c.watch_events,
        c.fork_events,
        c.release_events,

        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.push_events / c.all_retained_event_count, 2)
        END AS push_share_pct,
        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.pull_request_events / c.all_retained_event_count, 2)
        END AS pull_request_share_pct,
        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.issue_events / c.all_retained_event_count, 2)
        END AS issue_share_pct,
        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.issue_comment_events / c.all_retained_event_count, 2)
        END AS issue_comment_share_pct,
        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.review_events / c.all_retained_event_count, 2)
        END AS review_share_pct,
        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.inline_review_comment_events / c.all_retained_event_count, 2)
        END AS inline_review_comment_share_pct,
        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.watch_events / c.all_retained_event_count, 2)
        END AS watch_share_pct,
        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.fork_events / c.all_retained_event_count, 2)
        END AS fork_share_pct,
        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.release_events / c.all_retained_event_count, 2)
        END AS release_share_pct,

        c.actor_known_events,
        c.unique_known_actors,
        CASE WHEN c.all_retained_event_count > 0
             THEN ROUND(100 * c.actor_known_events / c.all_retained_event_count, 2)
        END AS actor_attribution_coverage_pct,

        '{run_id}'           AS _gold_run_id,
        current_timestamp()  AS _gold_updated_at

    FROM activity_counts c
    LEFT JOIN repo_names n
      ON  n.repo_id = c.repo_id
      AND n.activity_date = c.activity_date
""")
gold_rows.createOrReplaceTempView("gold_rows")

## 6. Stage

The result is written to the staging copy first. The Gold table is not touched yet.

Spark does its work only now, when the rows are written. If something goes wrong, the error is kept in `stage_error`, so that the audit row at the end can report it.

In [0]:
stage_error = None

try:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {STAGING_TABLE} AS
        SELECT * FROM gold_rows
    """)
    print("Staged:", STAGING_TABLE)

except Exception as error:
    stage_error = str(error)[:1000]
    print("Staging FAILED:", stage_error)

## 7. Quality gates

The checks run on the staging copy. One of them, the event IDs, runs on the stacked Silver rows. If a gate fails, nothing is published.

| Gate | It fails when |
|---|---|
| `KEY_NOT_UNIQUE` | A repository and day appear more than once |
| `KEY_IS_NULL` | A row has no `repo_id` or no `activity_date` |
| `COUNT_IS_NULL` | One of the 12 count columns is missing. A count is 0 when nothing was counted, never NULL |
| `TYPES_DO_NOT_ADD_UP` | The nine counts together are not the total |
| `SHARE_IS_NULL` | One of the nine shares is missing. A share is 0 when the kind has no events, never NULL |
| `SHARE_OUT_OF_RANGE` | A share is below 0 or above 100 |
| `SHARE_WITHOUT_EVENTS` | A kind has 0 events, but its share is not 0 |
| `SHARES_DO_NOT_ADD_UP` | The nine shares together are not 100 within rounding: below 99.95 or above 100.05 |
| `ACTOR_COUNTS_OUT_OF_ORDER` | Different actors ≤ events with a known actor ≤ all events is not true, or a row has events with a known actor but 0 actors |
| `COVERAGE_OUT_OF_RANGE` | A coverage is below 0 or above 100 |
| `COVERAGE_DOES_NOT_FIT_ITS_BASE` | A row with events has no coverage, or a row without events has one |
| `EVENT_ID_NOT_UNIQUE` | An `event_id` appears more than once in the nine Silver tables together, on the affected days |
| `EVENT_SUM_DIFFERS_FROM_SILVER` | The sum of one of the nine event columns is not the number of rows of its Silver table on the affected days |

### 7.1 Keys that appear more than once

In [0]:
duplicate_key_count = 0

if stage_error is None:
    duplicate_key_count = spark.sql(f"""
        SELECT COUNT(*) AS duplicate_keys
        FROM (
            SELECT repo_id, activity_date
            FROM {STAGING_TABLE}
            GROUP BY repo_id, activity_date
            HAVING COUNT(*) > 1
        ) keys_seen_twice
    """).collect()[0]["duplicate_keys"]

print("Keys that appear more than once:", duplicate_key_count)

### 7.2 Event IDs that appear more than once

Each Silver table has every `event_id` only once. Here the nine tables are stacked, so the same check is made on all nine together. If one event were in two tables, it would be counted twice in the total.

This check reads the stacked Silver rows of step 4, not the staging copy, because the Gold rows no longer carry event IDs. The profiling run found 33,623,791 rows and 33,623,791 different event IDs in the nine tables together.

In [0]:
duplicate_event_id_count = 0

if stage_error is None:
    duplicate_event_id_count = spark.sql("""
        SELECT COUNT(*) AS duplicate_event_ids
        FROM (
            SELECT event_id
            FROM activity_rows
            GROUP BY event_id
            HAVING COUNT(*) > 1
        ) event_ids_seen_twice
    """).collect()[0]["duplicate_event_ids"]

print("Event IDs that appear more than once:", duplicate_event_id_count)

### 7.3 The other checks

One query counts, for every rule, the rows that break it. The expected value of each count is 0. The last ten values are the row count and the sums of the nine event columns.

- **Shares add up.** Each share is rounded to 2 decimals, so it can be off by up to 0.005. Nine shares together can be off by up to 0.045. That is why the check allows 99.95 to 100.05.
- `COALESCE(SUM(...), 0)`: the sum of no rows is NULL. For the comparison with Silver it has to be 0.

The comparison with Silver is made nine times, once per table. The nine sums are put into a dictionary with the same names as `silver_row_counts` from step 4.1. Two dictionaries are equal when every name has the same number in both.

In [0]:
failed_checks = []
staged_row_count = 0
staged_event_sums = {
    "push_events": 0,
    "pull_request_events": 0,
    "issue_events": 0,
    "issue_comment_events": 0,
    "pull_request_review_events": 0,
    "pull_request_review_comment_events": 0,
    "watch_events": 0,
    "fork_events": 0,
    "release_events": 0,
}

if stage_error is None:
    gate_row = spark.sql(f"""
        SELECT
            count_if(repo_id IS NULL OR activity_date IS NULL)
                AS rows_without_key,

            count_if(all_retained_event_count IS NULL
                  OR push_events IS NULL
                  OR pull_request_events IS NULL
                  OR issue_events IS NULL
                  OR issue_comment_events IS NULL
                  OR review_events IS NULL
                  OR inline_review_comment_events IS NULL
                  OR watch_events IS NULL
                  OR fork_events IS NULL
                  OR release_events IS NULL
                  OR actor_known_events IS NULL
                  OR unique_known_actors IS NULL)
                AS rows_with_missing_count,

            count_if(push_events
                     + pull_request_events
                     + issue_events
                     + issue_comment_events
                     + review_events
                     + inline_review_comment_events
                     + watch_events
                     + fork_events
                     + release_events
                     <> all_retained_event_count)
                AS rows_where_types_do_not_add_up,

            count_if(push_share_pct IS NULL
                  OR pull_request_share_pct IS NULL
                  OR issue_share_pct IS NULL
                  OR issue_comment_share_pct IS NULL
                  OR review_share_pct IS NULL
                  OR inline_review_comment_share_pct IS NULL
                  OR watch_share_pct IS NULL
                  OR fork_share_pct IS NULL
                  OR release_share_pct IS NULL)
                AS rows_with_missing_share,

            count_if(push_share_pct < 0 OR push_share_pct > 100
                  OR pull_request_share_pct < 0 OR pull_request_share_pct > 100
                  OR issue_share_pct < 0 OR issue_share_pct > 100
                  OR issue_comment_share_pct < 0 OR issue_comment_share_pct > 100
                  OR review_share_pct < 0 OR review_share_pct > 100
                  OR inline_review_comment_share_pct < 0 OR inline_review_comment_share_pct > 100
                  OR watch_share_pct < 0 OR watch_share_pct > 100
                  OR fork_share_pct < 0 OR fork_share_pct > 100
                  OR release_share_pct < 0 OR release_share_pct > 100)
                AS rows_with_share_out_of_range,

            count_if((push_events = 0 AND push_share_pct <> 0)
                  OR (pull_request_events = 0 AND pull_request_share_pct <> 0)
                  OR (issue_events = 0 AND issue_share_pct <> 0)
                  OR (issue_comment_events = 0 AND issue_comment_share_pct <> 0)
                  OR (review_events = 0 AND review_share_pct <> 0)
                  OR (inline_review_comment_events = 0 AND inline_review_comment_share_pct <> 0)
                  OR (watch_events = 0 AND watch_share_pct <> 0)
                  OR (fork_events = 0 AND fork_share_pct <> 0)
                  OR (release_events = 0 AND release_share_pct <> 0))
                AS rows_with_share_without_events,

            count_if(ROUND(push_share_pct
                           + pull_request_share_pct
                           + issue_share_pct
                           + issue_comment_share_pct
                           + review_share_pct
                           + inline_review_comment_share_pct
                           + watch_share_pct
                           + fork_share_pct
                           + release_share_pct, 2) NOT BETWEEN 99.95 AND 100.05)
                AS rows_where_shares_do_not_add_up,

            count_if(unique_known_actors > actor_known_events
                  OR actor_known_events > all_retained_event_count
                  OR (actor_known_events > 0 AND unique_known_actors = 0))
                AS rows_with_actor_counts_out_of_order,

            count_if(actor_attribution_coverage_pct < 0
                  OR actor_attribution_coverage_pct > 100)
                AS rows_with_coverage_out_of_range,

            count_if((all_retained_event_count > 0 AND actor_attribution_coverage_pct IS NULL)
                  OR (all_retained_event_count = 0 AND actor_attribution_coverage_pct IS NOT NULL))
                AS rows_where_coverage_does_not_fit,

            COUNT(*)                                        AS staged_rows,
            COALESCE(SUM(push_events), 0)                   AS staged_push_sum,
            COALESCE(SUM(pull_request_events), 0)           AS staged_pull_request_sum,
            COALESCE(SUM(issue_events), 0)                  AS staged_issue_sum,
            COALESCE(SUM(issue_comment_events), 0)          AS staged_issue_comment_sum,
            COALESCE(SUM(review_events), 0)                 AS staged_review_sum,
            COALESCE(SUM(inline_review_comment_events), 0)  AS staged_review_comment_sum,
            COALESCE(SUM(watch_events), 0)                  AS staged_watch_sum,
            COALESCE(SUM(fork_events), 0)                   AS staged_fork_sum,
            COALESCE(SUM(release_events), 0)                AS staged_release_sum

        FROM {STAGING_TABLE}
    """).collect()[0]

    staged_row_count = gate_row["staged_rows"]

    # The sum of each Gold event column, under the name of its Silver table.
    staged_event_sums["push_events"] = gate_row["staged_push_sum"]
    staged_event_sums["pull_request_events"] = gate_row["staged_pull_request_sum"]
    staged_event_sums["issue_events"] = gate_row["staged_issue_sum"]
    staged_event_sums["issue_comment_events"] = gate_row["staged_issue_comment_sum"]
    staged_event_sums["pull_request_review_events"] = gate_row["staged_review_sum"]
    staged_event_sums["pull_request_review_comment_events"] = gate_row["staged_review_comment_sum"]
    staged_event_sums["watch_events"] = gate_row["staged_watch_sum"]
    staged_event_sums["fork_events"] = gate_row["staged_fork_sum"]
    staged_event_sums["release_events"] = gate_row["staged_release_sum"]

    if duplicate_key_count > 0:
        failed_checks.append("KEY_NOT_UNIQUE")
    if gate_row["rows_without_key"] > 0:
        failed_checks.append("KEY_IS_NULL")
    if gate_row["rows_with_missing_count"] > 0:
        failed_checks.append("COUNT_IS_NULL")
    if gate_row["rows_where_types_do_not_add_up"] > 0:
        failed_checks.append("TYPES_DO_NOT_ADD_UP")
    if gate_row["rows_with_missing_share"] > 0:
        failed_checks.append("SHARE_IS_NULL")
    if gate_row["rows_with_share_out_of_range"] > 0:
        failed_checks.append("SHARE_OUT_OF_RANGE")
    if gate_row["rows_with_share_without_events"] > 0:
        failed_checks.append("SHARE_WITHOUT_EVENTS")
    if gate_row["rows_where_shares_do_not_add_up"] > 0:
        failed_checks.append("SHARES_DO_NOT_ADD_UP")
    if gate_row["rows_with_actor_counts_out_of_order"] > 0:
        failed_checks.append("ACTOR_COUNTS_OUT_OF_ORDER")
    if gate_row["rows_with_coverage_out_of_range"] > 0:
        failed_checks.append("COVERAGE_OUT_OF_RANGE")
    if gate_row["rows_where_coverage_does_not_fit"] > 0:
        failed_checks.append("COVERAGE_DOES_NOT_FIT_ITS_BASE")
    if duplicate_event_id_count > 0:
        failed_checks.append("EVENT_ID_NOT_UNIQUE")
    if staged_event_sums != silver_row_counts:
        failed_checks.append("EVENT_SUM_DIFFERS_FROM_SILVER")

print("Staged rows (repository-days):", staged_row_count)
print("Sum of each Gold event column, next to the rows of its Silver table:")
for silver_table_name in silver_row_counts:
    print("  ", silver_table_name, ":", staged_event_sums[silver_table_name],
          "| Silver rows:", silver_row_counts[silver_table_name])

if len(failed_checks) == 0:
    print("Quality gates: no gate failed")
else:
    print("Quality gates FAILED:", "; ".join(failed_checks))

## 8. Publish

What happens with the staged rows:

| Situation | Status | Gold table |
|---|---|---|
| Staging failed | `FAILED` | Unchanged |
| There is no affected day, because Silver has no rows in the scope of this run | `NO_INPUT` | Unchanged |
| A quality gate failed | `FAILED` | Unchanged |
| Everything is fine | `SUCCEEDED` | The affected days are replaced by the staged rows |

How the days are replaced:

- **With a `source_date`:** `INSERT INTO ... REPLACE WHERE activity_date IN (...)` deletes the rows of the affected days and inserts the staged rows. The other days are not touched. Delta also refuses a staged row that belongs to another day.
- **With a blank date:** `INSERT OVERWRITE` replaces the whole table. This also removes a day that no longer exists in Silver.

Both are one Delta commit. A reader sees either the old rows or the new ones, never a half-written day. A repository-day that no longer has events disappears, because its whole day is replaced.

In [0]:
run_status = None
error_message = None
published_rows = 0

if stage_error is not None:
    run_status = "FAILED"
    error_message = "Staging failed: " + stage_error

elif len(affected_dates) == 0:
    run_status = "NO_INPUT"

elif len(failed_checks) > 0:
    run_status = "FAILED"
    error_message = "Quality gates failed: " + "; ".join(failed_checks)

else:
    # The first line of the statement depends on the scope of the run.
    if source_date_value is None:
        publish_start = f"INSERT OVERWRITE TABLE {GOLD_TABLE}"
    else:
        publish_start = (
            f"INSERT INTO {GOLD_TABLE} "
            f"REPLACE WHERE activity_date IN ({affected_days_sql})"
        )

    print("Publish statement starts with:", publish_start)

    try:
        spark.sql(f"""
            {publish_start}
            SELECT
                repo_id,
                activity_date,
                repo_name,
                all_retained_event_count,
                push_events,
                pull_request_events,
                issue_events,
                issue_comment_events,
                review_events,
                inline_review_comment_events,
                watch_events,
                fork_events,
                release_events,
                push_share_pct,
                pull_request_share_pct,
                issue_share_pct,
                issue_comment_share_pct,
                review_share_pct,
                inline_review_comment_share_pct,
                watch_share_pct,
                fork_share_pct,
                release_share_pct,
                actor_known_events,
                unique_known_actors,
                actor_attribution_coverage_pct,
                _gold_run_id,
                _gold_updated_at
            FROM {STAGING_TABLE}
        """)
        run_status = "SUCCEEDED"
        published_rows = staged_row_count

    except Exception as error:
        run_status = "FAILED"
        error_message = "Publish failed: " + str(error)[:1000]

print("Status:", run_status)
print("Published rows:", published_rows)

## 9. Quality numbers for the audit row

These totals describe what this run calculated. They are counted on the staging copy, so they are there also when the run was not published. None of them stops the run.

After a run for all dates they can be compared directly with the expected values in section 14 of the contract.

`unique_known_actors` summed over the rows counts a person once for every repository and day on which they took part. It is a number of person-repository-days, not of people.

In [0]:
quality_metrics = {
    "repository_days": 0,
    "all_retained_event_count": 0,
    "push_events": 0,
    "pull_request_events": 0,
    "issue_events": 0,
    "issue_comment_events": 0,
    "review_events": 0,
    "inline_review_comment_events": 0,
    "watch_events": 0,
    "fork_events": 0,
    "release_events": 0,
    "actor_known_events": 0,
    "unique_known_actors": 0,
}

if stage_error is None:
    total_row = spark.sql(f"""
        SELECT
            COUNT(*)                                        AS repository_days,
            COALESCE(SUM(all_retained_event_count), 0)      AS all_retained_event_count,
            COALESCE(SUM(push_events), 0)                   AS push_events,
            COALESCE(SUM(pull_request_events), 0)           AS pull_request_events,
            COALESCE(SUM(issue_events), 0)                  AS issue_events,
            COALESCE(SUM(issue_comment_events), 0)          AS issue_comment_events,
            COALESCE(SUM(review_events), 0)                 AS review_events,
            COALESCE(SUM(inline_review_comment_events), 0)  AS inline_review_comment_events,
            COALESCE(SUM(watch_events), 0)                  AS watch_events,
            COALESCE(SUM(fork_events), 0)                   AS fork_events,
            COALESCE(SUM(release_events), 0)                AS release_events,
            COALESCE(SUM(actor_known_events), 0)            AS actor_known_events,
            COALESCE(SUM(unique_known_actors), 0)           AS unique_known_actors
        FROM {STAGING_TABLE}
    """).collect()[0]

    quality_metrics["repository_days"] = total_row["repository_days"]
    quality_metrics["all_retained_event_count"] = total_row["all_retained_event_count"]
    quality_metrics["push_events"] = total_row["push_events"]
    quality_metrics["pull_request_events"] = total_row["pull_request_events"]
    quality_metrics["issue_events"] = total_row["issue_events"]
    quality_metrics["issue_comment_events"] = total_row["issue_comment_events"]
    quality_metrics["review_events"] = total_row["review_events"]
    quality_metrics["inline_review_comment_events"] = total_row["inline_review_comment_events"]
    quality_metrics["watch_events"] = total_row["watch_events"]
    quality_metrics["fork_events"] = total_row["fork_events"]
    quality_metrics["release_events"] = total_row["release_events"]
    quality_metrics["actor_known_events"] = total_row["actor_known_events"]
    quality_metrics["unique_known_actors"] = total_row["unique_known_actors"]

print("Repository-days                    :", quality_metrics["repository_days"])
print("All events                         :", quality_metrics["all_retained_event_count"])
print("  push                             :", quality_metrics["push_events"])
print("  pull request                     :", quality_metrics["pull_request_events"])
print("  issue                            :", quality_metrics["issue_events"])
print("  issue comment                    :", quality_metrics["issue_comment_events"])
print("  review                           :", quality_metrics["review_events"])
print("  inline review comment            :", quality_metrics["inline_review_comment_events"])
print("  watch                            :", quality_metrics["watch_events"])
print("  fork                             :", quality_metrics["fork_events"])
print("  release                          :", quality_metrics["release_events"])
print("Events with a known actor          :", quality_metrics["actor_known_events"])
print("Actors (person-repository-days)    :", quality_metrics["unique_known_actors"])

## 10. Audit

One row is added to the Gold audit table for this run. The values are written in the order of the table's columns.

`source_rows_read` has one number per Silver table: the dictionary of step 4.1.

If the run failed, the audit row is written first. Then the notebook stops with an error, so that the Job task fails too.

In [0]:
# The Delta version of the Gold table after this run.
gold_table_version = spark.sql(
    f"DESCRIBE HISTORY {GOLD_TABLE} LIMIT 1"
).collect()[0]["version"]

if len(failed_checks) == 0:
    failed_checks_text = None
else:
    failed_checks_text = "; ".join(failed_checks)

# True when no gate failed. A run without rows has no failed gate.
checks_passed = stage_error is None and len(failed_checks) == 0

run_finished_at = datetime.now(timezone.utc)

audit_row = (
    str(uuid.uuid4()),                                  # audit_id
    run_id,                                             # run_id
    PIPELINE_NAME,                                      # pipeline_name
    GOLD_TABLE,                                         # gold_table
    processing_scope,                                   # processing_scope
    source_date_value,                                  # source_date
    affected_dates,                                     # affected_dates
    run_started_at,                                     # run_started_at
    run_finished_at,                                    # run_finished_at
    run_status,                                         # status
    error_message,                                      # error_message
    silver_row_counts,                                  # source_rows_read
    {"missing_event_date": rows_without_event_date},    # rows_excluded
    published_rows,                                     # gold_rows_published
    gold_table_version,                                 # gold_table_version
    checks_passed,                                      # checks_passed
    failed_checks_text,                                 # failed_checks
    quality_metrics,                                    # quality_metrics
)

audit_schema = spark.table(AUDIT_TABLE).schema
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print("Audit row written:", AUDIT_TABLE)
print("run_id:", run_id)
print("status:", run_status)

if run_status == "NO_INPUT":
    print("WARNING: Silver has no rows in the scope of this run. The Gold table was not changed.")

if rows_without_event_date > 0:
    print("WARNING: rows without an event date that were left out:", rows_without_event_date)

if run_status == "FAILED":
    raise RuntimeError("repository_activity_profile FAILED: " + error_message)

## 11. Results

These cells only read. They show the audit row of this run and the Gold table as it is after this run.

### 11.1 The audit row of this run

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {AUDIT_TABLE}
    WHERE run_id = '{run_id}'
      AND pipeline_name = '{PIPELINE_NAME}'
"""))

### 11.2 The Gold table per day

Expected on the real week, from the profiling run of 14 November 2026:

| Day | Rows | All events |
|---|---|---|
| 2024-02-01 | 780,577 | 4,931,231 |
| 2024-02-02 | 735,149 | 4,944,140 |
| 2024-02-03 | 579,908 | 4,447,005 |
| 2024-02-04 | 602,850 | 4,441,124 |
| 2024-02-05 | 763,841 | 4,789,485 |
| 2024-02-06 | 777,905 | 4,999,343 |
| 2024-02-07 | 760,343 | 5,071,463 |

Three of the columns can be compared with the Gold tables that are already built. On every day, `pull_request_events` must be the pull request events of G1, `issue_events` the issue events of G2, and issue comments + reviews + inline review comments the collaboration events of G4.

In [0]:
display(spark.sql(f"""
    SELECT
        activity_date,
        COUNT(*)                           AS repository_days,
        SUM(all_retained_event_count)      AS all_events,
        SUM(push_events)                   AS push_events,
        SUM(pull_request_events)           AS pull_request_events,
        SUM(issue_events)                  AS issue_events,
        SUM(issue_comment_events)          AS issue_comment_events,
        SUM(review_events)                 AS review_events,
        SUM(inline_review_comment_events)  AS inline_review_comment_events,
        SUM(watch_events)                  AS watch_events,
        SUM(fork_events)                   AS fork_events,
        SUM(release_events)                AS release_events,
        SUM(actor_known_events)            AS actor_known_events,
        SUM(unique_known_actors)           AS actors
    FROM {GOLD_TABLE}
    GROUP BY activity_date
    ORDER BY activity_date
"""))

### 11.3 The whole Gold table in one row

Expected after a run for all dates on the real week. The sum of each event column is the number of rows of its Silver table:

| Value | Expected |
|---|---|
| Rows | 5,000,573 |
| All events | 33,623,791 |
| Push events | 26,536,967 |
| Pull request events | 2,173,958 |
| Issue events | 548,956 |
| Issue comment events | 1,357,324 |
| Review events | 717,277 |
| Inline review comment events | 422,917 |
| Watch events | 1,380,559 |
| Fork events | 326,417 |
| Release events | 159,416 |
| Events with a known actor | 33,623,791, so the coverage of the whole week is 100 |

The number of actors was not measured before. It cannot be above the number of events with a known actor.

The shares in this cell are the shares of the whole week. They are calculated from the sums, not as the average of the shares of the rows. Pushes should be about 78.9% of all events.

In [0]:
display(spark.sql(f"""
    SELECT
        COUNT(*)                           AS repository_days,
        COUNT(DISTINCT activity_date)      AS days,
        SUM(all_retained_event_count)      AS all_events,
        SUM(push_events)                   AS push_events,
        SUM(pull_request_events)           AS pull_request_events,
        SUM(issue_events)                  AS issue_events,
        SUM(issue_comment_events)          AS issue_comment_events,
        SUM(review_events)                 AS review_events,
        SUM(inline_review_comment_events)  AS inline_review_comment_events,
        SUM(watch_events)                  AS watch_events,
        SUM(fork_events)                   AS fork_events,
        SUM(release_events)                AS release_events,
        SUM(actor_known_events)            AS actor_known_events,
        SUM(unique_known_actors)           AS actors,
        CASE
            WHEN SUM(all_retained_event_count) > 0
            THEN ROUND(100 * SUM(push_events) / SUM(all_retained_event_count), 2)
        END                                AS push_share_pct_of_all_rows,
        CASE
            WHEN SUM(all_retained_event_count) > 0
            THEN ROUND(100 * SUM(actor_known_events) / SUM(all_retained_event_count), 2)
        END                                AS actor_coverage_pct_of_all_rows
    FROM {GOLD_TABLE}
"""))

### 11.4 A sample: the ten repository-days with the most events

A look at real rows. The busiest rows belong to automated repositories. The profiling run found one repository with 64,783 events on a single day.

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {GOLD_TABLE}
    ORDER BY all_retained_event_count DESC, repo_id, activity_date
    LIMIT 10
"""))

## What this notebook wrote

- `gold.repository_activity_profile`: the affected days were replaced, when the status is `SUCCEEDED`.
- `gold_staging.repository_activity_profile`: the staging copy, overwritten on every run.
- `audit.gold_pipeline_runs`: one new row.

With a `test_schema`, all three are in that schema.

## How to read the table

- The table shows a mix of event counts. It does not measure the amount of work or the health of a project.
- Watch events are new stars only. Removed stars are not in the data.
- Pushes are 78.9% of all events, so most repository-days are mostly pushes.
- A share of 0.00 next to a count above 0 is possible: one event among many thousands rounds to 0.00. The count columns are the exact values.
- 63% of the rows are built from one or two events. The count columns allow a reader to set a threshold.
- The repositories with the most events are automated.